# Руководство по дообучению для нового воплощения

Этот блокнот — руководство по дообучению предобученной модели GR00T-N1 на новом датасете.

# 1. Руководство по дообучению на Lerobot SO100

GR00T-N1 доступна для роботов разной конструкции. Недорогой [манипулятор So100 Lerobot](https://github.com/huggingface/lerobot/blob/main/examples/10_use_so100.md) от Hugging Face позволяет пользователям дообучить GR00T-N1 на собственном роботе с помощью тега `new_embodiment`.


![so100_eval_demo.gif](../media/so100_eval_demo.gif)

## Шаг 1: датасет

Для дообучения можно использовать любой датасет lerobot. В этом руководстве сначала используем пример датасета: [so100_strawberry_grape](https://huggingface.co/spaces/lerobot/visualize_dataset?dataset=youliangtan%2Fso100_strawberry_grape&episode=0)

Обратите внимание, что это воплощение не использовалось в смеси предобучающих датасетов.


### Сначала скачайте датасет

```bash
huggingface-cli download --repo-type dataset youliangtan/so100_strawberry_grape --local-dir ./demo_data/so100_strawberry_grape
```

### Затем скопируйте файл модальностей

Файл `modality.json` содержит дополнительную информацию о модальностях состояния и действия, благодаря чему датасет становится совместимым с GR00T. Скопируйте `examples/so100__modality.json` в `<DATASET_PATH>/meta/modality.json`.

```bash
cp examples/so100__modality.json ./demo_data/so100_strawberry_grape/meta/modality.json
```

Затем можно загрузить датасет с помощью класса `LeRobotSingleDataset`.

In [ ]:
from gr00t.utils.misc import any_describe
from gr00t.data.dataset import LeRobotSingleDataset
from gr00t.experiment.data_config import DATA_CONFIG_MAP

dataset_path = "./demo_data/so100_strawberry_grape"   # change this to your dataset path

data_config = DATA_CONFIG_MAP["so100"]

dataset = LeRobotSingleDataset(
    dataset_path=dataset_path,
    modality_configs=data_config.modality_config(),
    embodiment_tag="new_embodiment",
    video_backend="torchvision_av",
)

resp = dataset[7]
any_describe(resp)

In [ ]:
# visualize the dataset
# show img
import matplotlib.pyplot as plt

images_list = []

for i in range(100):
    if i % 10 == 0:
        resp = dataset[i]
        img = resp["video.webcam"][0]
        images_list.append(img)

fig, axs = plt.subplots(2, 5, figsize=(20, 10))
for i, ax in enumerate(axs.flat):
    ax.imshow(images_list[i])
    ax.axis("off")
    ax.set_title(f"Image {i}")
plt.tight_layout() # adjust the subplots to fit into the figure area.
plt.show()

## Шаг 2: дообучение

Дообучение выполняется скриптом `scripts/gr00t_finetune.py`.


```bash
python scripts/gr00t_finetune.py \
   --dataset-path /datasets/so100_strawberry_grape/ \
   --num-gpus 1 \
   --output-dir ~/so100-checkpoints  \
   --max-steps 2000 \
   --data-config so100 \
   --video-backend torchvision_av
```

## Шаг 3: оценка в открытом цикле

После обучения можно выполнить следующую команду, чтобы визуализировать дообученную политику.

```bash
python scripts/eval_policy.py --plot \
   --embodiment_tag new_embodiment \
   --model_path <YOUR_CHECKPOINT_PATH> \
   --data_config so100 \
  --dataset_path /datasets/so100_strawberry_grape/ \
   --video_backend torchvision_av \
   --modality_keys single_arm gripper
```

Это график после 7000 шагов обучения политики.

![so100-7k-steps.png](../media/so100-7k-steps.png)


После большего числа шагов обучения график будет заметно лучше.


Отлично! GR00T-N1 успешно дообучена на новом воплощении.

## Развёртывание

Подробнее о развёртывании см. в файле: `5-policy-deployment.md`.

---

# 2. Руководство по дообучению на датасете укладки блоков G1

Здесь пошагово описано, как дообучить GR00T-N1 на датасете укладки блоков G1.

## Шаг 1: датасет

Загрузка любого датасета для дообучения выполняется в два шага:
- 1.1: определить конфигурацию модальностей и преобразования для датасета
- 1.2: загрузить датасет с помощью класса `LeRobotSingleDataset`

### Шаг 1.0: скачать датасет

- Скачайте датасет по адресу: https://huggingface.co/datasets/unitreerobotics/G1_BlockStacking_Dataset
- Скопируйте `examples/unitree_g1_blocks__modality.json` в `<DATASET_PATH>/meta/modality.json`
  - Файл содержит дополнительную информацию о модальностях состояния и действия, благодаря чему датасет становится совместимым с GR00T
  - `cp examples/unitree_g1_blocks__modality.json datasets/G1_BlockStacking_Dataset/meta/modality.json`


**Разбор конфигурации модальностей**

Файл содержит подробные метаданные о модальностях состояния и действия, что позволяет:

- **Разделить хранение данных и их интерпретацию:**
  - **Состояние и действие:** хранятся как объединённые массивы float32. Файл `modality.json` содержит метаданные, нужные, чтобы интерпретировать эти массивы как отдельные детализированные поля с дополнительной информацией для обучения.
  - **Видео:** хранятся отдельными файлами; конфигурационный файл позволяет переименовать их в стандартный формат.
  - **Аннотации:** отслеживаются все поля аннотаций. Если аннотаций нет, не включайте поле `annotation` в конфигурационный файл.
- **Детальное разбиение:** массивы состояния и действия делятся на поля с более понятным смыслом.
- **Явное сопоставление:** явное соответствие измерений данных.
- **Сложные преобразования данных:** во время обучения поддерживаются нормализация и поворот для отдельных полей.

#### Схема

```json
{
    "state": {
        "<state_name>": {
            "start": <int>,         // начальный индекс в массиве состояния
            "end": <int>,           // конечный индекс в массиве состояния
        }
    },
    "action": {
        "<action_name>": {
            "start": <int>,         // начальный индекс в массиве действий
            "end": <int>,           // конечный индекс в массиве действий
        }
    },
    "video": {
        "<video_name>": {}  // пустой словарь для единообразия с другими модальностями
    },
    "annotation": {
        "<annotation_name>": {}  // пустой словарь для единообразия с другими модальностями
    }
}
```

Пример приведён в `getting_started/examples/unitree_g1_blocks__modality.json`. Этот файл находится в папке `meta` датасета lerobot.


Статистику (`meta/metadata.json`) создаём командой:
```bash ```bash
python scripts/load_dataset.py --data_path /datasets/G1_BlockStacking_Dataset/ --embodiment_tag new_embodiment
```

In [ ]:
from gr00t.data.schema import EmbodimentTag

In [ ]:
dataset_path = "./demo_data/g1"  # change this to your dataset path
embodiment_tag = EmbodimentTag.NEW_EMBODIMENT

### Шаг 1.1: конфигурация модальностей и преобразования

Конфигурация модальностей позволяет выбрать, какие конкретные потоки данных использовать при дообучении для каждого типа входа (видео, состояние, действие, язык и т. д.), и точно управлять тем, какие части датасета задействованы.

In [ ]:
from gr00t.data.dataset import ModalityConfig


# select the modality keys you want to use for finetuning
video_modality = ModalityConfig(
    delta_indices=[0],
    modality_keys=["video.cam_right_high"],
)

state_modality = ModalityConfig(
    delta_indices=[0],
    modality_keys=["state.left_arm", "state.right_arm", "state.left_hand", "state.right_hand"],
)

action_modality = ModalityConfig(
    delta_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    modality_keys=["action.left_arm", "action.right_arm", "action.left_hand", "action.right_hand"],
)

language_modality = ModalityConfig(
    delta_indices=[0],
    modality_keys=["annotation.human.task_description"],
)

modality_configs = {
    "video": video_modality,
    "state": state_modality,
    "action": action_modality,
    "language": language_modality,
}

In [ ]:
from gr00t.data.transform.base import ComposedModalityTransform
from gr00t.data.transform import VideoToTensor, VideoCrop, VideoResize, VideoColorJitter, VideoToNumpy
from gr00t.data.transform.state_action import StateActionToTensor, StateActionTransform
from gr00t.data.transform.concat import ConcatTransform
from gr00t.model.transforms import GR00TTransform


# select the transforms you want to apply to the data
to_apply_transforms = ComposedModalityTransform(
    transforms=[
        # video transforms
        VideoToTensor(apply_to=video_modality.modality_keys, backend="torchvision"),
        VideoCrop(apply_to=video_modality.modality_keys, scale=0.95, backend="torchvision"),
        VideoResize(apply_to=video_modality.modality_keys, height=224, width=224, interpolation="linear", backend="torchvision" ),
        VideoColorJitter(apply_to=video_modality.modality_keys, brightness=0.3, contrast=0.4, saturation=0.5, hue=0.08, backend="torchvision"),
        VideoToNumpy(apply_to=video_modality.modality_keys),

        # state transforms
        StateActionToTensor(apply_to=state_modality.modality_keys),
        StateActionTransform(apply_to=state_modality.modality_keys, normalization_modes={
            "state.left_arm": "min_max",
            "state.right_arm": "min_max",
            "state.left_hand": "min_max",
            "state.right_hand": "min_max",
        }),

        # action transforms
        StateActionToTensor(apply_to=action_modality.modality_keys),
        StateActionTransform(apply_to=action_modality.modality_keys, normalization_modes={
            "action.right_arm": "min_max",
            "action.left_arm": "min_max",
            "action.right_hand": "min_max",
            "action.left_hand": "min_max",
        }),

        # ConcatTransform
        ConcatTransform(
            video_concat_order=video_modality.modality_keys,
            state_concat_order=state_modality.modality_keys,
            action_concat_order=action_modality.modality_keys,
        ),
        # model-specific transform
        GR00TTransform(
            state_horizon=len(state_modality.delta_indices),
            action_horizon=len(action_modality.delta_indices),
            max_state_dim=64,
            max_action_dim=32,
        ),
    ]
)


### Шаг 1.2: загрузка датасета

Сначала визуализируем датасет, а затем загрузим его с помощью класса `LeRobotSingleDataset` (без преобразований).

In [ ]:
from gr00t.data.dataset import LeRobotSingleDataset

train_dataset = LeRobotSingleDataset(
    dataset_path=dataset_path,
    modality_configs=modality_configs,
    embodiment_tag=embodiment_tag,
    video_backend="torchvision_av",
)


In [ ]:
# use matplotlib to visualize the images
import matplotlib.pyplot as plt
import numpy as np

print(train_dataset[0].keys())

images = []
for i in range(5):
    image = train_dataset[i]["video.cam_right_high"][0]
    # image is in HWC format, convert it to CHW format
    image = image.transpose(2, 0, 1)
    images.append(image)   

fig, axs = plt.subplots(1, 5, figsize=(20, 5))
for i, image in enumerate(images):
    axs[i].imshow(np.transpose(image, (1, 2, 0)))
    axs[i].axis("off")
plt.show()

Теперь инициализируем датасет с нашей конфигурацией модальностей и преобразованиями.

In [ ]:
train_dataset = LeRobotSingleDataset(
    dataset_path=dataset_path,
    modality_configs=modality_configs,
    embodiment_tag=embodiment_tag,
    video_backend="torchvision_av",
    transforms=to_apply_transforms,
)

**Дополнительно**:
 - Для ускорения обучения используется загрузчик с кешем: он загружает все данные в память, что заметно повышает производительность. Но если датасет очень большой или возникает ошибка нехватки памяти (OOM), можно перейти на стандартный загрузчик lerobot (`gr00t.data.dataset.LeRobotSingleDataset`). Он использует тот же API, поэтому переключаться между ними можно без изменения кода.
 - В качестве видеобэкенда используется torchvision_av: видео кодируется через av, а не стандартным h264.


### Шаг 2: загрузка модели

Процесс обучения делится на три шага:
- 2.1: загрузить базовую модель из Hugging Face или по локальному пути
- 2.2: подготовить параметры обучения
- 2.3: запустить цикл обучения

#### Шаг 2.1: загрузка базовой модели

Модель загружаем методом `from_pretrained_for_tuning`. Он позволяет указать, какие части модели подлежат настройке.

In [ ]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from gr00t.model.gr00t_n1 import GR00T_N1

BASE_MODEL_PATH = "nvidia/GR00T-N1-2B"
TUNE_LLM = False            # Whether to tune the LLM
TUNE_VISUAL = True          # Whether to tune the visual encoder
TUNE_PROJECTOR = True       # Whether to tune the projector
TUNE_DIFFUSION_MODEL = True # Whether to tune the diffusion model

model = GR00T_N1.from_pretrained(
    pretrained_model_name_or_path=BASE_MODEL_PATH,
    tune_llm=TUNE_LLM,  # backbone's LLM
    tune_visual=TUNE_VISUAL,  # backbone's vision tower
    tune_projector=TUNE_PROJECTOR,  # action head's projector
    tune_diffusion_model=TUNE_DIFFUSION_MODEL,  # action head's DiT
)

# Set the model's compute_dtype to bfloat16
model.compute_dtype = "bfloat16"
model.config.compute_dtype = "bfloat16"
model.to(device)

#### Шаг 2.2: подготовка параметров обучения

Для настройки обучения используем `TrainingArguments` из Hugging Face. Основные параметры:

In [ ]:
from transformers import TrainingArguments

output_dir = "output/model/path"    # CHANGE THIS ACCORDING TO YOUR LOCAL PATH
per_device_train_batch_size = 8     # CHANGE THIS ACCORDING TO YOUR GPU MEMORY
max_steps = 20                      # CHANGE THIS ACCORDING TO YOUR NEEDS
report_to = "wandb"
dataloader_num_workers = 8

training_args = TrainingArguments(
    output_dir=output_dir,
    run_name=None,
    remove_unused_columns=False,
    deepspeed="",
    gradient_checkpointing=False,
    bf16=True,
    tf32=True,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=1,
    dataloader_num_workers=dataloader_num_workers,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=True,
    optim="adamw_torch",
    adam_beta1=0.95,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    learning_rate=1e-4,
    weight_decay=1e-5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10.0,
    num_train_epochs=300,
    max_steps=max_steps,
    save_strategy="steps",
    save_steps=500,
    evaluation_strategy="no",
    save_total_limit=8,
    report_to=report_to,
    seed=42,
    do_eval=False,
    ddp_find_unused_parameters=False,
    ddp_bucket_cap_mb=100,
    torch_compile_mode=None,
)


#### Шаг 2.3: инициализация тренера и запуск цикла обучения

In [ ]:
from gr00t.experiment.runner import TrainRunner

experiment = TrainRunner(
    train_dataset=train_dataset,
    model=model,
    training_args=training_args,
)

experiment.train()

Ниже сравниваются результаты офлайн-проверки после 1000 шагов и после 10000 шагов:

**Результаты дообучения на датасете укладки блоков Unitree G1:**

| 1k шагов | 10k шагов |
| --- | --- |
| ![1k](../media/g1_ft_1k.png) | ![10k](../media/g1_ft_10k.png) |
| MSE: 0.0181 | MSE: 0.0022 | 